# Level 2 — Task 1: Predicting House Prices with Linear Regression

## Objective
Build a machine-learning model that predicts house prices from housing characteristics using **Linear Regression**.

## Dataset
This project uses the public **California Housing** dataset available through Scikit-learn. It contains 20,640 observations from California districts and eight numerical housing/economic features.

Target:
- `MedHouseVal` — median house value in units of $100,000.

Features:
- `MedInc` — median income
- `HouseAge` — median house age
- `AveRooms` — average rooms
- `AveBedrms` — average bedrooms
- `Population`
- `AveOccup` — average household occupancy
- `Latitude`
- `Longitude`

## Workflow
1. Load and inspect the housing data.
2. Perform exploratory analysis.
3. Check missing values.
4. Split data into training and testing sets.
5. Train a Linear Regression model.
6. Evaluate with MAE, RMSE and R².
7. Compare actual vs predicted prices.
8. Analyse model coefficients.
9. Draw business/real-estate conclusions.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Download/load the public dataset through scikit-learn
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Shape:", df.shape)
display(df.head())


## 1. Initial inspection


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDescriptive statistics:")
display(df.describe().T)


In [ ]:
plt.figure(figsize=(9, 6))
sns.histplot(df["MedHouseVal"], bins=40, kde=True)
plt.title("Distribution of Median House Value")
plt.xlabel("Median House Value ($100,000 units)")
plt.ylabel("Number of Districts")
plt.show()


**Observation:** House values are not perfectly normally distributed. Some areas have substantially higher values than the majority, so model errors should be evaluated with more than one metric.


## 2. Correlation analysis


In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()


**Observation:** Median income (`MedInc`) generally has the strongest linear relationship with median house value among the available features. Geographic variables such as latitude and longitude also contain useful information, although their relationship is not necessarily purely linear.


## 3. Feature-target relationship


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=df.sample(min(5000, len(df)), random_state=42),
    x="MedInc", y="MedHouseVal", alpha=0.4, ax=axes[0]
)
axes[0].set_title("Median Income vs House Value")

sns.scatterplot(
    data=df.sample(min(5000, len(df)), random_state=42),
    x="HouseAge", y="MedHouseVal", alpha=0.4, ax=axes[1]
)
axes[1].set_title("House Age vs House Value")

plt.tight_layout()
plt.show()


## 4. Train-test split

The target is `MedHouseVal`. The data is split into 80% training and 20% testing observations. The test set is kept separate until final evaluation.


In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Train Linear Regression model


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Value": [mae, rmse, r2]
})

display(metrics)


### Metric interpretation

- **MAE:** average absolute prediction error.
- **RMSE:** penalizes larger errors more strongly than MAE.
- **R²:** proportion of target variance explained by the linear model.

Because the target is measured in $100,000 units, MAE and RMSE can be multiplied by 100,000 to express them approximately in dollars.


In [ ]:
print(f"MAE:  {mae:.4f} (${mae*100000:,.0f})")
print(f"RMSE: {rmse:.4f} (${rmse*100000:,.0f})")
print(f"R²:   {r2:.4f}")


## 6. Actual vs predicted prices


In [ ]:
plt.figure(figsize=(8, 7))
sns.scatterplot(x=y_test, y=pred, alpha=0.45)

line_min = min(y_test.min(), pred.min())
line_max = max(y_test.max(), pred.max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle="--")

plt.xlabel("Actual House Value ($100,000 units)")
plt.ylabel("Predicted House Value ($100,000 units)")
plt.title("Actual vs Predicted House Values")
plt.show()


**Observation:** Points close to the diagonal line represent accurate predictions. Larger vertical distances indicate larger prediction errors.


## 7. Model coefficients

Coefficients show the direction and magnitude of each feature's relationship with the target while holding the other features constant. Their raw magnitudes should be interpreted carefully because the features have different units.


In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", key=abs, ascending=False)

display(coef_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x="Coefficient", y="Feature")
plt.title("Linear Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("")
plt.show()


## 8. Residual analysis


In [ ]:
residuals = y_test - pred

plt.figure(figsize=(9, 5))
sns.scatterplot(x=pred, y=residuals, alpha=0.4)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted House Value")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()


**Observation:** A well-behaved residual plot should not show a strong systematic pattern. Visible patterns or changing spread can indicate that a simple linear model is missing nonlinear relationships.


## 9. Business interpretation

### Key insights
- Income is an important predictor of housing value in this dataset.
- Location-related features contribute useful information.
- Linear Regression provides an interpretable baseline model.
- Prediction errors show that house prices are influenced by relationships that are more complex than a purely linear equation.

### Practical use
A real-estate business could use this type of model as a baseline for estimating property values, screening listings, and identifying potentially underpriced or overpriced areas.

### Limitations
- This dataset represents California districts rather than individual house listings.
- The model is linear and may miss nonlinear interactions.
- Predictions should not be treated as professional property appraisals.
- A production system should be validated on current, local market data.


In [ ]:
# Save metrics and coefficients
metrics.to_csv("../output/model_metrics.csv", index=False)
coef_df.to_csv("../output/model_coefficients.csv", index=False)

# Save a sample of predictions
prediction_output = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": pred,
    "Residual": residuals.values
})
prediction_output.to_csv("../output/predictions.csv", index=False)

print("House-price Linear Regression task completed.")


## Conclusion

A complete Linear Regression workflow was implemented from data loading and EDA through model training and evaluation. MAE, RMSE and R² were used together to evaluate performance, while actual-vs-predicted and residual plots were used to understand model behaviour.

The model is an interpretable baseline. More advanced models such as Random Forest, Gradient Boosting or other nonlinear methods could be compared in a future improvement stage.
